# CNN for Alzheimer's Disease Classification from 3D MRI
## Complete Training Notebook for Google Colab

이 노트북은 3D CNN을 사용하여 뇌 MRI에서 알츠하이머병(AD), 경도인지장애(MCI), 정상(CN)을 분류하는 모델을 학습합니다.

**논문**: Generalizable deep learning model for early Alzheimer's disease detection from structural MRIs  
**데이터셋**: ADNI (Alzheimer's Disease Neuroimaging Initiative)

---

## 1. 라이브러리 설치 및 Import

In [ ]:
# 필요한 라이브러리 설치
!pip install nibabel scipy pandas pyyaml scikit-learn scikit-image matplotlib progress

In [ ]:
# Import statements
import os
import sys
import shutil
import time
import math
import random
import warnings
import traceback
from pathlib import Path
from itertools import cycle

import numpy as np
import pandas as pd
import scipy
import scipy.ndimage
from scipy.ndimage.interpolation import shift, rotate, zoom
from skimage.transform import resize

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch import cat
import torch.nn.init as init
import torch.utils.data as data
from torch.optim.lr_scheduler import MultiStepLR
from torch.autograd import Variable, Function

import nibabel as nib
from sklearn.preprocessing import label_binarize
from sklearn.metrics import accuracy_score, roc_curve, auc, confusion_matrix

import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Configuration 설정

**중요**: `dir_to_scans`와 `dir_to_tsv` 경로를 실제 데이터 경로로 변경해야 합니다!

In [ ]:
# Configuration (config.yaml 내용)
cfg = {
    'file_name': './saved_model/colab_training',
    'exp_name': 'colab_experiment',
    
    'data': {
        'data_root_dir': 'data/',
        'dir_to_scans': '/path/to/your/scans',  # ⚠️ 실제 MRI 스캔 경로로 변경 필요!
        'dir_to_tsv': './tsv_files',  # ⚠️ TSV 파일 경로로 변경 필요!
        'batch_size': 4,
        'val_batch_size': 2,
        'workers': 2,  # Colab에서는 낮게 설정
        'percentage_usage': 1.0
    },
    
    'model': {
        'arch': 'ours',
        'input_channel': 1,
        'nhid': 512,
        'feature_dim': 1024,
        'n_label': 3,  # CN, MCI, AD
        'expansion': 8,  # 네트워크 너비 (논문에서 최적값)
        'num_blocks': 0,
        'type_name': 'conv3x3x3',
        'norm_type': 'Instance'  # Instance Normalization
    },
    
    'adv_model': {
        'nhid': 36,
        'out_dim': 12
    },
    
    'training_parameters': {
        'use_age': False,  # Age encoding 사용 여부
        'pretrain': None,
        'start_epoch': 0,
        'epochs': 200,  # 총 에폭 수
        'print_freq': 10,
        'max_grad_l2_norm': None
    },
    
    'optimizer': {
        'method': 'SGD',
        'par': {
            'lr': 0.01,
            'weight_decay': 0.0
        }
    },
    
    'visdom': {
        'port': None,
        'server': None  # Visdom 비활성화 (Colab에서는 사용 안 함)
    }
}

# 저장 디렉토리 생성
os.makedirs('./saved_model', exist_ok=True)

print("Configuration loaded:")
print(f"  - Epochs: {cfg['training_parameters']['epochs']}")
print(f"  - Batch size: {cfg['data']['batch_size']}")
print(f"  - Expansion: {cfg['model']['expansion']}")
print(f"  - Learning rate: {cfg['optimizer']['par']['lr']}")

## 3. 유틸리티 함수들

In [ ]:
# ===== AverageMeter =====
class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


# ===== FullModel for DataParallel =====
class FullModel(nn.Module):
    def __init__(self, model, loss):
        super(FullModel, self).__init__()
        self.model = model
        self.loss = loss
    
    def forward(self, inputs, targets):
        outputs = self.model(inputs[0], inputs[1])
        loss = self.loss(outputs, targets)
        return torch.unsqueeze(loss, 0), outputs


def DataParallel_withLoss(model, loss, **kwargs):
    """Wrap model with loss for multi-GPU training"""
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
    model = FullModel(model, loss)
    device_ids = kwargs.get('device_ids', None)
    output_device = kwargs.get('output_device', None)
    model = torch.nn.DataParallel(model, device_ids=device_ids, output_device=output_device).to(device)
    return model


def clip_gradients(myModel, i_iter, max_grad_l2_norm):
    """Clip gradients to prevent explosion"""
    if max_grad_l2_norm is not None:
        nn.utils.clip_grad_norm_(myModel.parameters(), max_grad_l2_norm)


def balanced_accuracy_score(y_true, y_pred, sample_weight=None, adjusted=False):
    """Calculate balanced accuracy (평균 per-class accuracy)"""
    C = confusion_matrix(y_true, y_pred, sample_weight=sample_weight)
    with np.errstate(divide='ignore', invalid='ignore'):
        per_class = np.diag(C) / C.sum(axis=1)
    if np.any(np.isnan(per_class)):
        warnings.warn('y_pred contains classes not in y_true')
        per_class = per_class[~np.isnan(per_class)]
    score = np.mean(per_class)
    if adjusted:
        n_classes = len(per_class)
        chance = 1 / n_classes
        score -= chance
        score /= 1 - chance
    return score


def get_auc_data(logit_all, target_all, n_label):
    """Calculate ROC-AUC for multi-class classification"""
    y = label_binarize(target_all, classes=list(range(n_label)))
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    
    # Per-class AUC
    for k in range(n_label):
        fpr[k], tpr[k], _ = roc_curve(y[:, k], logit_all[:, k])
        roc_auc[k] = auc(fpr[k], tpr[k])
    
    # Micro-average AUC
    fpr["micro"], tpr["micro"], _ = roc_curve(y.ravel(), logit_all.ravel())
    roc_auc[k+1] = auc(fpr["micro"], tpr["micro"])
    
    # Macro-average AUC
    all_fpr = np.unique(np.concatenate([fpr[k] for k in range(n_label)]))
    mean_tpr = np.zeros_like(all_fpr)
    for k in range(n_label):
        mean_tpr += np.interp(all_fpr, fpr[k], tpr[k])
    mean_tpr /= n_label
    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_auc[k+2] = auc(fpr["macro"], tpr["macro"])
    
    # Prepare plotting data
    plotting_fpr = [fpr[k] for k in range(n_label)] + [fpr["micro"], fpr["macro"]]
    plotting_tpr = [tpr[k] for k in range(n_label)] + [tpr["micro"], tpr["macro"]]
    
    return plotting_fpr, plotting_tpr, roc_auc


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)
    
    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))
    
    res = []
    correct_all = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0, keepdim=True)
        correct_all.append(correct_k.clone())
        res.append(correct_k.mul_(100.0 / batch_size))
    return res, correct_all

print("✓ Utility functions loaded")

## 4. Loss Functions

In [ ]:
def get_loss_criterion(loss_config, type):
    """Get loss function based on type"""
    if type == 'CrossEntropyLoss':
        loss_criterion = nn.CrossEntropyLoss(ignore_index=-100)
    elif type == 'mse':
        loss_criterion = nn.MSELoss()
    else:
        raise NotImplementedError(f"Loss type '{type}' not implemented")
    return loss_criterion

print("✓ Loss functions loaded")

## 5. Data Augmentation Functions

In [ ]:
# 3D Image augmentation functions

def translateit(image, offset, isseg=False):
    """3D translation"""
    order = 0 if isseg else 5
    return shift(image, (int(offset[0]), int(offset[1]), int(offset[2])), order=order, mode='constant')


def scaleit(image, factor, isseg=False):
    """3D scaling"""
    order = 0 if isseg else 3
    height, width, depth = image.shape
    zheight = int(np.round(factor * height))
    zwidth = int(np.round(factor * width))
    zdepth = depth
    
    if factor < 1.0:
        newimg = np.zeros_like(image)
        row = (height - zheight) // 2
        col = (width - zwidth) // 2
        layer = (depth - zdepth) // 2
        newimg[row:row+zheight, col:col+zwidth, layer:layer+zdepth] = zoom(
            image, (float(factor), float(factor), 1.0), order=order, mode='nearest'
        )[0:zheight, 0:zwidth, 0:zdepth]
        return newimg
    elif factor > 1.0:
        row = (zheight - height) // 2
        col = (zwidth - width) // 2
        layer = (zdepth - depth) // 2
        newimg = zoom(
            image[row:row+zheight, col:col+zwidth, layer:layer+zdepth],
            (float(factor), float(factor), 1.0), order=order, mode='nearest'
        )
        extrah = (newimg.shape[0] - height) // 2
        extraw = (newimg.shape[1] - width) // 2
        extrad = (newimg.shape[2] - depth) // 2
        newimg = newimg[extrah:extrah+height, extraw:extraw+width, extrad:extrad+depth]
        return newimg
    else:
        return image


def rotateit(image, axes, theta, isseg=False):
    """3D rotation"""
    order = 0 if isseg else 5
    return rotate(image, float(theta), axes=axes, reshape=False, order=order, mode='constant')


def flipit(image):
    """Random flipping"""
    lr_thred = np.random.uniform(0, 1, 1)[0]
    ud_thred = np.random.uniform(0, 1, 1)[0]
    if lr_thred <= 0.5:
        image = np.fliplr(image)
    if ud_thred >= 0.5:
        image = np.flipud(image)
    return image


def intensifyit(image, factor):
    """Intensity adjustment"""
    return image * float(factor)


def resampleit(image, dims, isseg=False):
    """Resampling"""
    order = 0 if isseg else 5
    image = zoom(image, np.array(dims)/np.array(image.shape, dtype=np.float32), 
                 order=order, mode='nearest')
    if image.shape[-1] == 3:  # RGB image
        return image
    else:
        return image if isseg else (image - image.min()) / (image.max() - image.min())

print("✓ Augmentation functions loaded")

## 6. Model Architecture

In [ ]:
# ===== Age Encoding Module =====
class AgeEncoding(nn.Module):
    """Age encoding using positional encoding (선택적 사용)"""
    def __init__(self, d_model, dropout, out_dim, max_len=240):
        super(AgeEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Positional encoding
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp((torch.arange(0, d_model, 2).float() *
                             -(math.log(10000.0) / d_model)).float())
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
        
        self.fc6 = nn.Sequential(
            nn.Linear(d_model, 512),
            nn.LayerNorm(512),
            nn.Linear(512, out_dim)
        )
    
    def forward(self, x, age_id):
        y = torch.autograd.Variable(self.pe[age_id, :], requires_grad=False)
        y = self.fc6(y)
        x += y
        return self.dropout(x)


# ===== 3D CNN Backbone =====
class NetWork(nn.Module):
    """3D CNN backbone for MRI feature extraction"""
    def __init__(self, in_channel=1, feat_dim=1024, expansion=4, 
                 type_name='conv3x3x3', norm_type='Instance'):
        super(NetWork, self).__init__()
        
        self.conv = nn.Sequential()
        
        # Block 1: 1x1x1 conv
        self.conv.add_module('conv0_s1', nn.Conv3d(in_channel, 4*expansion, kernel_size=1))
        if norm_type == 'Instance':
            self.conv.add_module('lrn0_s1', nn.InstanceNorm3d(4*expansion))
        else:
            self.conv.add_module('lrn0_s1', nn.BatchNorm3d(4*expansion))
        self.conv.add_module('relu0_s1', nn.ReLU(inplace=True))
        self.conv.add_module('pool0_s1', nn.MaxPool3d(kernel_size=3, stride=2))
        
        # Block 2: 3x3x3 conv with dilation
        self.conv.add_module('conv1_s1', nn.Conv3d(4*expansion, 32*expansion, 
                                                   kernel_size=3, padding=0, dilation=2))
        if norm_type == 'Instance':
            self.conv.add_module('lrn1_s1', nn.InstanceNorm3d(32*expansion))
        else:
            self.conv.add_module('lrn1_s1', nn.BatchNorm3d(32*expansion))
        self.conv.add_module('relu1_s1', nn.ReLU(inplace=True))
        self.conv.add_module('pool1_s1', nn.MaxPool3d(kernel_size=3, stride=2))
        
        # Block 3: 5x5x5 conv with dilation
        self.conv.add_module('conv2_s1', nn.Conv3d(32*expansion, 64*expansion, 
                                                   kernel_size=5, padding=2, dilation=2))
        if norm_type == 'Instance':
            self.conv.add_module('lrn2_s1', nn.InstanceNorm3d(64*expansion))
        else:
            self.conv.add_module('lrn2_s1', nn.BatchNorm3d(64*expansion))
        self.conv.add_module('relu2_s1', nn.ReLU(inplace=True))
        self.conv.add_module('pool2_s1', nn.MaxPool3d(kernel_size=3, stride=2))
        
        # Block 4: 3x3x3 conv with dilation
        self.conv.add_module('conv3_s1', nn.Conv3d(64*expansion, 64*expansion, 
                                                   kernel_size=3, padding=1, dilation=2))
        if norm_type == 'Instance':
            self.conv.add_module('lrn3_s1', nn.InstanceNorm3d(64*expansion))
        else:
            self.conv.add_module('lrn3_s1', nn.BatchNorm3d(64*expansion))
        self.conv.add_module('relu3_s1', nn.ReLU(inplace=True))
        self.conv.add_module('pool3_s1', nn.MaxPool3d(kernel_size=5, stride=2))
        
        # Fully connected layer
        self.fc6 = nn.Sequential(
            nn.Linear(64*expansion*5*5*5, feat_dim)
        )
        
        # Age encoder (optional)
        self.age_encoder = AgeEncoding(512, 0.1, feat_dim)
    
    def forward(self, x, age_id):
        z = self.conv(x)
        z = self.fc6(z.view(x.shape[0], -1))
        if age_id is not None:
            z = self.age_encoder(z, age_id)
        return z

print("✓ Model architecture (NetWork) loaded")

In [ ]:
# ===== Classifier Head =====
class Flatten(nn.Module):
    def __init__(self):
        super(Flatten, self).__init__()
    
    def forward(self, feat):
        return feat.view(feat.size(0), -1)


class LinearClassifierAlexNet(nn.Module):
    """2-layer classifier for AD/MCI/CN classification"""
    def __init__(self, in_dim, n_hid=200, n_label=3):
        super(LinearClassifierAlexNet, self).__init__()
        
        self.classifier = nn.Sequential(
            Flatten(),
            nn.Linear(in_dim, n_hid),
            nn.Linear(n_hid, n_label)
        )
        self.initilize()
    
    def initilize(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                m.weight.data.normal_(0, 0.01)
                m.bias.data.fill_(0.0)
    
    def forward(self, x):
        return self.classifier(x)

print("✓ Classifier loaded")

In [ ]:
# ===== Complete Model Wrapper =====
class alex_net_complete(nn.Module):
    """Complete model: backbone + classifier"""
    def __init__(self, image_embedding_model, classifier=None):
        super(alex_net_complete, self).__init__()
        self.image_embedding_model = image_embedding_model
        self.classifier = classifier
    
    def forward(self, input_image_variable, age_id=None):
        image_embedding = self.image_embedding_model(input_image_variable, age_id)
        if self.classifier is None:
            logit_res = image_embedding
        else:
            logit_res = self.classifier(image_embedding)
        return logit_res

print("✓ Complete model wrapper loaded")

In [ ]:
# ===== Model Builder =====
def prepare_model(arch='ours', in_channel=1, feat_dim=128, n_hid_main=200, 
                  n_label=3, out_dim=128, expansion=8, type_name='conv3x3x3', 
                  norm_type='Instance'):
    """Prepare complete model"""
    if arch == 'ours':
        image_embeding_model = NetWork(in_channel=in_channel, feat_dim=feat_dim, 
                                       expansion=expansion, type_name=type_name, 
                                       norm_type=norm_type)
    else:
        raise ValueError('Wrong Architecture!')
    
    classifier = LinearClassifierAlexNet(in_dim=feat_dim, n_hid=n_hid_main, n_label=n_label)
    main_model = alex_net_complete(image_embeding_model, classifier)
    
    return main_model


def build_model(config):
    """Build model from config"""
    arch = config['model']['arch']
    in_channel = config['model']['input_channel']
    feat_dim = config['model']['feature_dim']
    n_label = config['model']['n_label']
    n_hid_main = config['model']['nhid']
    out_dim = config['adv_model']['out_dim']
    expansion = config['model']['expansion']
    type_name = config['model']['type_name']
    norm_type = config['model']['norm_type']
    
    main_model = prepare_model(arch=arch, in_channel=in_channel, feat_dim=feat_dim,
                               n_hid_main=n_hid_main, n_label=n_label, out_dim=out_dim,
                               expansion=expansion, type_name=type_name, norm_type=norm_type)
    
    # Load pretrained weights if specified
    if config['training_parameters']['pretrain'] is not None:
        best_model_dir = './saved_model/'
        pretrained_dict = torch.load(
            best_model_dir + config['training_parameters']['pretrain'] + '_model_low_loss.pth.tar',
            map_location='cpu'
        )['state_dict']
        model_dict = main_model.state_dict()
        pretrained_dict = {k[6:]: v for k, v in pretrained_dict.items() if (k[6:] in model_dict.keys())}
        model_dict.update(pretrained_dict)
        main_model.load_state_dict(model_dict)
        print("Pretrained weights loaded!")
    
    return main_model

print("✓ Model builder loaded")

## 7. Dataset Loader

In [ ]:
class ADNI_3D(data.Dataset):
    """ADNI 3D MRI Dataset Loader"""
    
    def __init__(self, dir_to_scans, dir_to_tsv, mode='Train', n_label=3, percentage_usage=1.0):
        # Label mapping
        if n_label == 3:
            LABEL_MAPPING = ["CN", "MCI", "AD"]
        elif n_label == 2:
            LABEL_MAPPING = ["CN", "AD"]
        self.LABEL_MAPPING = LABEL_MAPPING
        
        # Load TSV file
        tsv_file = os.path.join(dir_to_tsv, mode + '_diagnosis_ADNI.tsv')
        subject_tsv = pd.read_csv(tsv_file, sep='\t')
        
        # Clean sessions without labels
        indices_not_missing = []
        for i in range(len(subject_tsv)):
            if subject_tsv.iloc[i].diagnosis in LABEL_MAPPING:
                indices_not_missing.append(i)
        
        self.subject_tsv = subject_tsv.iloc[indices_not_missing]
        
        # Use percentage of data for training
        if mode == 'Train':
            n_samples = int(len(self.subject_tsv) * percentage_usage)
            self.subject_tsv = self.subject_tsv.iloc[np.random.permutation(n_samples)]
        
        self.subject_id = np.unique(subject_tsv.participant_id.values)
        self.index_dic = dict(zip(self.subject_id, range(len(self.subject_id))))
        self.dir_to_scans = dir_to_scans
        self.mode = mode
        self.age_range = list(np.arange(0.0, 120.0, 0.5))
    
    def __len__(self):
        return len(self.subject_tsv)
    
    def __getitem__(self, idx):
        try:
            # Construct path to scan
            path = os.path.join(
                self.dir_to_scans,
                self.subject_tsv.iloc[idx].participant_id,
                self.subject_tsv.iloc[idx].session_id,
                't1/spm/segmentation/normalized_space'
            )
            all_segs = list(os.listdir(path))
            
            # Get label
            diagnosis = self.subject_tsv.iloc[idx].diagnosis
            if diagnosis == 'CN':
                label = 0
            elif diagnosis == 'MCI':
                label = 1
            elif diagnosis == 'AD':
                label = 1 if self.LABEL_MAPPING == ["CN", "AD"] else 2
            else:
                print('WRONG LABEL VALUE!!!')
                label = -100
            
            # Get metadata
            mmse = self.subject_tsv.iloc[idx].mmse
            cdr_sub = 0
            age = self.age_range.index(self.subject_tsv.iloc[idx].age_rounded)
            idx_out = self.index_dic[self.subject_tsv.iloc[idx].participant_id]
            
            # Load MRI scan
            for seg_name in all_segs:
                if 'Space_T1w' in seg_name:
                    image = nib.load(os.path.join(path, seg_name)).get_fdata().squeeze()
                    image[np.isnan(image)] = 0.0
                    # Normalize to [0, 1]
                    image = (image - image.min()) / (image.max() - image.min() + 1e-6)
                    
                    # Data augmentation for training
                    if self.mode == 'Train':
                        image = self.augment_image(image)
            
            image = np.expand_dims(image, axis=0)
            
            # Crop to 96x96x96
            if self.mode == 'Train':
                image = self.randomCrop(image, 96, 96, 96)
            else:
                image = self.centerCrop(image, 96, 96, 96)
        
        except Exception as e:
            print(f"Failed to load #{idx}: {path}")
            print(f"Errors encountered: {e}")
            print(traceback.format_exc())
            return None, None, None, None, None, None
        
        return image.astype(np.float32), label, idx_out, mmse, cdr_sub, age
    
    def centerCrop(self, img, length, width, height):
        """Center crop to target size"""
        assert img.shape[1] >= length
        assert img.shape[2] >= width
        assert img.shape[3] >= height
        
        x = img.shape[1] // 2 - length // 2
        y = img.shape[2] // 2 - width // 2
        z = img.shape[3] // 2 - height // 2
        img = img[:, x:x+length, y:y+width, z:z+height]
        return img
    
    def randomCrop(self, img, length, width, height):
        """Random crop to target size"""
        assert img.shape[1] >= length
        assert img.shape[2] >= width
        assert img.shape[3] >= height
        
        x = random.randint(0, img.shape[1] - length)
        y = random.randint(0, img.shape[2] - width)
        z = random.randint(0, img.shape[3] - height)
        img = img[:, x:x+length, y:y+width, z:z+height]
        return img
    
    def augment_image(self, image):
        """Apply Gaussian filtering augmentation"""
        sigma = np.random.uniform(0.0, 1.0, 1)[0]
        image = scipy.ndimage.filters.gaussian_filter(image, sigma, truncate=8)
        return image

print("✓ Dataset loader (ADNI_3D) loaded")

## 8. Training Functions

In [ ]:
def train(cfg, train_loader, main_model, scheduler, criterion, main_optimizer, epoch):
    """Train for one epoch"""
    batch_time = AverageMeter()
    data_time = AverageMeter()
    main_losses = AverageMeter()
    
    # Switch to train mode
    main_model.train()
    end = time.time()
    
    logit_all = []
    target_all = []
    
    for i, (input, target, index, mmse, segment, age) in enumerate(train_loader):
        # Measure data loading time
        data_time.update(time.time() - end)
        
        # Move to device
        input = input.to(device)
        target = target.to(device)
        index = index.to(device)
        age = age.to(device) if cfg['training_parameters']['use_age'] else None
        
        # Forward pass
        main_loss, logit = main_model([input, age], target)
        main_loss = main_loss.mean()
        
        # Store predictions
        logit_all.append(logit.data.cpu())
        target_all.append(target.data.cpu())
        acc, _ = accuracy(logit.data.cpu(), target.data.cpu())
        
        # Backward pass
        main_optimizer.zero_grad()
        main_loss.backward()
        clip_gradients(main_model, i, cfg['training_parameters']['max_grad_l2_norm'])
        main_optimizer.step()
        
        # Record loss
        main_losses.update(main_loss.cpu().item(), input.size(0))
        
        # Measure elapsed time
        batch_time.update(time.time() - end)
        end = time.time()
        
        # Print progress
        if i % cfg['training_parameters']['print_freq'] == 0:
            print(f'Epoch: [{epoch}][{i}/{len(train_loader)}]\t'
                  f'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  f'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  f'Loss {main_losses.val:.4f} ({main_losses.avg:.4f})\t'
                  f'Accuracy {acc[0].item():.3f}')
    
    # Calculate balanced accuracy
    logit_all = torch.cat(logit_all).numpy()
    target_all = torch.cat(target_all).numpy()
    acc_all = balanced_accuracy_score(target_all, np.argmax(logit_all, 1))
    
    return main_losses.avg, acc_all * 100


def validate(cfg, val_loader, main_model, criterion, epoch):
    """Validate the model"""
    batch_time = AverageMeter()
    data_time = AverageMeter()
    main_losses = AverageMeter()
    
    end = time.time()
    correct_all = 0.0
    
    # Switch to eval mode
    main_model.eval()
    
    confusion_mat = torch.zeros(cfg['model']['n_label'], cfg['model']['n_label'])
    logit_all = []
    target_all = []
    
    with torch.no_grad():
        for i, (input, target, patient_idx, mmse, segment, age) in enumerate(val_loader):
            # Measure data loading time
            data_time.update(time.time() - end)
            
            # Move to device
            input = input.to(device)
            target = target.to(device)
            age = age.to(device) if cfg['training_parameters']['use_age'] else None
            
            # Forward pass
            main_loss, logit = main_model([input, age], target)
            main_loss = main_loss.mean()
            
            # Store predictions
            logit_all.append(torch.tensor(logit.data.cpu()))
            target_all.append(torch.tensor(target.data.cpu()))
            
            # Calculate accuracy
            acc, _ = accuracy(logit.data.cpu(), target.data.cpu())
            _, preds = torch.max(logit.cpu(), 1)
            
            # Update confusion matrix
            for t, p in zip(target.cpu().view(-1), preds.view(-1)):
                confusion_mat[t.long(), p.long()] += 1
            
            acc, correct = accuracy(logit.cpu(), target.cpu())
            correct_all += correct[0].item()
            
            # Record loss
            main_losses.update(main_loss.cpu().item(), input.size(0))
            
            # Measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()
            
            # Print progress
            if i % cfg['training_parameters']['print_freq'] == 0:
                print(f'Validation [{i}/{len(val_loader)}]\t'
                      f'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                      f'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                      f'Loss {main_losses.val:.4f} ({main_losses.avg:.4f})\t'
                      f'Accuracy {acc[0].item():.3f}')
    
    # Calculate metrics
    logit_all = torch.cat(logit_all).numpy()
    target_all = torch.cat(target_all).numpy()
    acc_all = balanced_accuracy_score(target_all, np.argmax(logit_all, 1))
    plotting_fpr, plotting_tpr, roc_auc = get_auc_data(logit_all, target_all, cfg['model']['n_label'])
    
    return main_losses.avg, acc_all * 100, confusion_mat, [plotting_fpr, plotting_tpr, roc_auc]


def save_checkpoint(state, is_best, lowest_loss, is_best_micro_auc, is_best_macro_auc, 
                    is_best_auc, filename='checkpoint.pth.tar'):
    """Save model checkpoint"""
    saving_dir = Path(filename).parent
    if not os.path.exists(saving_dir):
        os.makedirs(saving_dir)
    
    torch.save(state, filename)
    
    if is_best:
        shutil.copyfile(filename, filename.replace('.pth.tar', '') + '_model_best.pth.tar')
    if lowest_loss:
        shutil.copyfile(filename, filename.replace('.pth.tar', '') + '_model_low_loss.pth.tar')
    if is_best_micro_auc:
        shutil.copyfile(filename, filename.replace('.pth.tar', '') + '_model_best_micro.pth.tar')
    if is_best_macro_auc:
        shutil.copyfile(filename, filename.replace('.pth.tar', '') + '_model_best_macro.pth.tar')
    if is_best_auc:
        shutil.copyfile(filename, filename.replace('.pth.tar', '') + '_model_best_auc.pth.tar')

print("✓ Training functions loaded")

## 9. Main Training Loop

In [ ]:
def main(cfg):
    """Main training function"""
    global best_prec1, best_loss, best_micro_auc, best_macro_auc
    
    best_prec1 = 0
    best_loss = 1000
    best_micro_auc = 0
    best_macro_auc = 0
    
    # Set seeds for reproducibility
    seed = 168
    torch.manual_seed(seed)
    if torch.cuda.device_count() > 0:
        torch.cuda.manual_seed(seed)
    
    print("\n" + "="*60)
    print("Building model...")
    print("="*60)
    
    # Build model
    main_model = build_model(cfg)
    main_model = main_model.to(device)
    print(f"Model built successfully! Parameters: {sum(p.numel() for p in main_model.parameters())/1e6:.2f}M")
    
    # Loss function
    criterion = get_loss_criterion(cfg, type='CrossEntropyLoss').to(device)
    
    # Wrap model with loss for DataParallel
    model = DataParallel_withLoss(main_model, criterion)
    
    if hasattr(model, 'module'):
        print('Has module!')
        model = model.module
    
    # Setup optimizer
    params = [{'params': filter(lambda p: p.requires_grad, model.parameters())}]
    main_optim = getattr(optim, cfg['optimizer']['method'])(params, **cfg['optimizer']['par'])
    
    # Learning rate scheduler (decay at epochs 20 and 50)
    scheduler = MultiStepLR(main_optim, milestones=[20, 50], gamma=0.1)
    
    print("\n" + "="*60)
    print("Loading data...")
    print("="*60)
    
    # Load datasets
    dir_to_scans = cfg['data']['dir_to_scans']
    dir_to_tsv = cfg['data']['dir_to_tsv']
    
    train_dataset = ADNI_3D(
        dir_to_scans, dir_to_tsv, mode='Train',
        n_label=cfg['model']['n_label'],
        percentage_usage=cfg['data']['percentage_usage']
    )
    
    val_dataset = ADNI_3D(
        dir_to_scans, dir_to_tsv, mode='Val',
        n_label=cfg['model']['n_label']
    )
    
    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=cfg['data']['batch_size'],
        shuffle=True, num_workers=cfg['data']['workers'],
        pin_memory=True, drop_last=True
    )
    
    val_loader = torch.utils.data.DataLoader(
        val_dataset, batch_size=cfg['data']['val_batch_size'],
        shuffle=False, num_workers=cfg['data']['workers'],
        pin_memory=True
    )
    
    ndata = len(train_dataset.subject_id)
    print(f'In total {ndata} patients in training set')
    print(f'Training batches: {len(train_loader)}')
    print(f'Validation batches: {len(val_loader)}')
    
    print("\n" + "="*60)
    print("Starting training...")
    print("="*60 + "\n")
    
    # Training loop
    for epoch in range(cfg['training_parameters']['start_epoch'], cfg['training_parameters']['epochs']):
        print(f"\n{'='*60}")
        print(f"Epoch [{epoch+1}/{cfg['training_parameters']['epochs']}]")
        print(f"{'='*60}")
        
        # Train for one epoch
        train_loss, train_acc = train(cfg, train_loader, model, scheduler, criterion, main_optim, epoch)
        
        # Validate
        val_loss, val_acc, confusion_matrix, auc_outs = validate(cfg, val_loader, model, criterion, epoch)
        
        # Learning rate scheduling
        scheduler.step()
        
        # Print epoch summary
        print(f"\n{'='*60}")
        print(f"Epoch [{epoch+1}] Summary:")
        print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"  Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
        print(f"  Micro AUC: {auc_outs[2][len(auc_outs[2])-2]:.4f}")
        print(f"  Macro AUC: {auc_outs[2][len(auc_outs[2])-1]:.4f}")
        print(f"{'='*60}\n")
        
        # Check for best model
        is_best = (val_acc > best_prec1)
        lowest_loss = (val_loss < best_loss)
        is_best_micro_auc = (auc_outs[2][len(auc_outs[2])-2] >= best_micro_auc)
        is_best_macro_auc = (auc_outs[2][len(auc_outs[2])-1] > best_macro_auc)
        
        best_prec1 = max(val_acc, best_prec1)
        best_loss = min(val_loss, best_loss)
        best_micro_auc = max(auc_outs[2][len(auc_outs[2])-2], best_micro_auc)
        best_macro_auc = max(auc_outs[2][len(auc_outs[2])-1], best_macro_auc)
        
        is_best_auc = (auc_outs[2][len(auc_outs[2])-2] > 0.8) & (auc_outs[2][len(auc_outs[2])-1] > 0.8)
        
        # Save checkpoint
        save_checkpoint(
            {
                'epoch': epoch + 1,
                'state_dict': model.state_dict(),
                'best_prec1': best_prec1,
                'best_loss': best_loss,
                'optimizer': main_optim.state_dict(),
            },
            is_best, lowest_loss, is_best_micro_auc, is_best_macro_auc, is_best_auc,
            filename=cfg['file_name'] + '.pth.tar'
        )
        
        if is_best:
            print(f"✓ New best accuracy: {val_acc:.2f}%")
        if is_best_micro_auc:
            print(f"✓ New best micro AUC: {auc_outs[2][len(auc_outs[2])-2]:.4f}")
        if is_best_macro_auc:
            print(f"✓ New best macro AUC: {auc_outs[2][len(auc_outs[2])-1]:.4f}")
    
    print("\n" + "="*60)
    print("Training completed!")
    print(f"Best Accuracy: {best_prec1:.2f}%")
    print(f"Best Micro AUC: {best_micro_auc:.4f}")
    print(f"Best Macro AUC: {best_macro_auc:.4f}")
    print("="*60)

print("✓ Main training loop loaded")

## 10. 실행하기

### ⚠️ 실행 전 필수 설정:

1. **데이터 경로 설정**: 위의 Configuration 셀에서 `dir_to_scans`와 `dir_to_tsv`를 실제 데이터 경로로 변경하세요.

2. **TSV 파일 준비**: `dir_to_tsv` 폴더에 다음 파일들이 필요합니다:
   - `Train_diagnosis_ADNI.tsv`
   - `Val_diagnosis_ADNI.tsv`
   - `Test_diagnosis_ADNI.tsv` (선택)

3. **MRI 스캔 구조**: 스캔은 다음 구조여야 합니다:
   ```
   dir_to_scans/
   ├── participant_id_1/
   │   └── session_id/
   │       └── t1/spm/segmentation/normalized_space/
   │           └── *Space_T1w*.nii.gz
   ├── participant_id_2/
   ...
   ```

### Google Drive 마운트 (Colab 사용 시):

```python
from google.colab import drive
drive.mount('/content/drive')

# 데이터 경로 업데이트
cfg['data']['dir_to_scans'] = '/content/drive/MyDrive/your_data/scans'
cfg['data']['dir_to_tsv'] = '/content/drive/MyDrive/your_data/tsv_files'
```

In [ ]:
# ⚠️ 데이터 경로가 올바르게 설정되었는지 확인!
print("Current configuration:")
print(f"Scans directory: {cfg['data']['dir_to_scans']}")
print(f"TSV directory: {cfg['data']['dir_to_tsv']}")
print(f"\nPlease verify these paths before running training!")

In [ ]:
# 학습 시작!
# ⚠️ 실행 전 위의 데이터 경로를 반드시 확인하세요!

if __name__ == '__main__':
    main(cfg)

## 11. 학습된 모델 로드 및 평가

학습이 완료된 후, 저장된 모델을 로드하여 평가할 수 있습니다.

In [ ]:
# 학습된 모델 로드
def load_trained_model(checkpoint_path, cfg):
    """Load trained model from checkpoint"""
    model = build_model(cfg)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Remove 'module.' prefix if exists
    state_dict = checkpoint['state_dict']
    new_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith('module.'):
            new_state_dict[k[7:]] = v
        else:
            new_state_dict[k] = v
    
    model.load_state_dict(new_state_dict)
    model = model.to(device)
    model.eval()
    
    print(f"Model loaded from {checkpoint_path}")
    print(f"Epoch: {checkpoint['epoch']}")
    print(f"Best accuracy: {checkpoint['best_prec1']:.2f}%")
    
    return model

# 예시: 최고 정확도 모델 로드
# model = load_trained_model('./saved_model/colab_training_model_best.pth.tar', cfg)

## 12. 하이퍼파라미터 조정 가이드

### 주요 하이퍼파라미터:

1. **Expansion (모델 너비)**:
   - 기본값: 8 (논문에서 최적)
   - 작은 데이터셋: 4 시도
   - 큰 데이터셋: 16 시도

2. **Learning Rate**:
   - 기본값: 0.01
   - 불안정한 학습: 0.001로 감소
   
3. **Batch Size**:
   - GPU 메모리에 따라 조정
   - Colab (16GB): 4-8
   - V100 (32GB): 8-16

4. **Epochs**:
   - 기본값: 200
   - 빠른 실험: 50
   - 수렴 확인 후 조기 종료

5. **Age Encoding**:
   - `use_age: True`로 설정하면 나이 정보 사용
   - 약간의 성능 향상 기대 (논문: +2% balanced acc)

### 메모리 부족 시:
```python
cfg['data']['batch_size'] = 2
cfg['data']['val_batch_size'] = 1
cfg['model']['expansion'] = 4  # 모델 크기 감소
```

## 13. 예상 결과 (논문 기준)

### ADNI Test Set:
- **Accuracy**: ~68%
- **Balanced Accuracy**: ~70%
- **Micro AUC**: ~82%
- **Macro AUC**: ~80%

### 클래스별 AUC:
- **CN (정상)**: ~88%
- **MCI (경도인지장애)**: ~63%
- **AD (알츠하이머)**: ~89%

### 학습 시간:
- Colab GPU: ~10-15시간 (200 epochs)
- V100: ~6-8시간

---

**참고 논문**:
- Gupta et al. "Generalizable deep learning model for early Alzheimer's disease detection from structural MRIs" 
- Scientific Reports (2022)
- Machine Learning for Health Workshop, PMLR (2020)